# Layer 3 of 4 - Transformer training

Consumes the splits from Layer 1, then defines the fixed architecture, optimizer schedule, and checkpoint-resumable training loop. The current baseline trains the context-32, seed-0 reference; completed run records include the fixed-target validation BPC consumed by Layer 4.


In [ ]:
import json
import math
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F


current = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (current, *current.parents) if (path / ".git").exists()), current)
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RUNS_DIR = RESULTS_DIR / "runs"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
for directory in (DATA_DIR, RESULTS_DIR, RUNS_DIR, CHECKPOINT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
DATA_PATHS = {name: DATA_DIR / f"{name}.txt" for name in ("train", "val", "test")}
missing = [path for path in DATA_PATHS.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Run layers/01_data/01_data_preparation.ipynb first; missing: " + ", ".join(map(str, missing)))
train_text, val_text, test_text = (DATA_PATHS[name].read_text(encoding="utf-8") for name in ("train", "val", "test"))
vocab = sorted(set(train_text))
char_to_id = {char: i for i, char in enumerate(vocab)}
def encode(split_text):
    return torch.tensor([char_to_id[char] for char in split_text])
train_data, val_data, test_data = (encode(split_text) for split_text in (train_text, val_text, test_text))
VOCAB_SIZE = len(vocab)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
EVAL_START = 256


## 2. Experiment settings

Only `context_len` changes between conditions. Every step sees 16,384 training characters, so the batch size is 512, 128 or 64.

In [ ]:
@dataclass
class Config:
    context_len: int
    seed: int
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 192
    dropout: float = 0.1
    chars_per_step: int = 16384
    max_steps: int = 3000
    lr: float = 1e-3
    min_lr: float = 1e-4
    weight_decay: float = 0.1
    grad_clip: float = 1.0
    eval_interval: int = 250
    eval_batches: int = 20

    @property
    def batch_size(self):
        return self.chars_per_step // self.context_len

    @property
    def run_name(self):
        return f"ctx{self.context_len}_seed{self.seed}"

## 3. Transformer

A GPT-style decoder: pre-LayerNorm blocks, causal self-attention, a GELU MLP, learned position embeddings, and output weights tied to the token embeddings.

The position table is created last, and the seed is reset just before the weights are initialised. With the same seed, every weight except the position table therefore starts identical at every context length.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head = config.n_head
        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.out = nn.Linear(config.n_embd, config.n_embd)
        self.attn_dropout = config.dropout
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        batch, length, width = x.shape
        q, k, v = (t.view(batch, length, self.n_head, -1).transpose(1, 2) for t in self.qkv(x).split(width, dim=2))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.attn_dropout if self.training else 0.0)
        return self.resid_dropout(self.out(y.transpose(1, 2).reshape(batch, length, width)))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn_norm = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.mlp_norm = nn.LayerNorm(config.n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        return x + self.mlp(self.mlp_norm(x))


def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(module.weight, std=0.02)
    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)


class CharTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embedding = nn.Embedding(VOCAB_SIZE, config.n_embd)
        self.embedding_dropout = nn.Dropout(config.dropout)
        self.blocks = nn.Sequential(*[Block(config) for _ in range(config.n_layer)])
        self.final_norm = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, VOCAB_SIZE, bias=False)
        self.head.weight = self.token_embedding.weight
        self.position_embedding = nn.Embedding(config.context_len, config.n_embd)
        torch.manual_seed(config.seed)
        self.apply(init_weights)

    def forward(self, ids):
        positions = torch.arange(ids.size(1), device=ids.device)
        x = self.embedding_dropout(self.token_embedding(ids) + self.position_embedding(positions))
        return self.head(self.final_norm(self.blocks(x)))

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

In [ ]:
for context_len in [32, 128, 256]:
    print(f"context {context_len}: {CharTransformer(Config(context_len, seed=0)).num_params():,} parameters")

short_model = CharTransformer(Config(32, seed=0)).state_dict()
long_model = CharTransformer(Config(256, seed=0)).state_dict()
assert all(torch.equal(short_model[name], long_model[name]) for name in short_model if name != "position_embedding.weight")
print("Same seed gives identical starting weights apart from the position table")

## 5. Metric: fixed-target bits-per-character

Every model scores the same targets: every character from position 257 of a split onward. A model with context length L predicts each target from the L characters before it, and only the last position of each window is scored. BPC = cross-entropy in nats / ln 2.

In [ ]:
def transformer_bpc(model, data, context_len, batch_size=512):
    model.eval()
    targets = torch.arange(EVAL_START, len(data))
    window = torch.arange(-context_len, 0)
    total_nats = 0.0
    for batch_targets in targets.split(batch_size):
        logits = model(data[batch_targets[:, None] + window].to(DEVICE))[:, -1]
        total_nats += F.cross_entropy(logits, data[batch_targets].to(DEVICE), reduction="sum").item()
    return total_nats / len(targets) / math.log(2)


## 7. Training loop

AdamW with a cosine learning-rate schedule from 1e-3 down to 1e-4, gradient clipping at 1.0, and fp16 mixed precision on GPU. Every `eval_interval` steps the loop:

- estimates the validation loss on a fixed set of validation batches;
- keeps the checkpoint with the lowest validation loss as `best.pt`;
- saves the full training state to `last.pt`, so a run that disconnects picks up where it stopped.

In [ ]:
def get_batch(data, context_len, batch_size, generator):
    starts = torch.randint(len(data) - context_len, (batch_size,), generator=generator)
    positions = starts[:, None] + torch.arange(context_len)
    return data[positions].to(DEVICE), data[positions + 1].to(DEVICE)


def loss_on(model, inputs, targets):
    with torch.autocast(DEVICE, dtype=torch.float16, enabled=USE_AMP):
        logits = model(inputs)
    return F.cross_entropy(logits.flatten(0, 1).float(), targets.flatten())


@torch.no_grad()
def validation_loss(model, config):
    model.eval()
    generator = torch.Generator().manual_seed(0)
    losses = [loss_on(model, *get_batch(val_data, config.context_len, config.batch_size, generator)).item()
              for _ in range(config.eval_batches)]
    model.train()
    return sum(losses) / len(losses)


def train(config):
    run_dir = CHECKPOINT_DIR / config.run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    last_path, best_path = run_dir / "last.pt", run_dir / "best.pt"

    model = CharTransformer(config).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.max_steps, eta_min=config.min_lr)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    batch_generator = torch.Generator().manual_seed(config.seed)
    progress = {"step": 0, "best_step": 0, "best_val_loss": math.inf, "train_seconds": 0.0, "history": []}

    if last_path.exists():
        checkpoint = torch.load(last_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        scaler.load_state_dict(checkpoint["scaler"])
        batch_generator.set_state(checkpoint["batch_generator"])
        progress = checkpoint["progress"]
        print(f"Resuming {config.run_name} from step {progress['step']}")

    torch.manual_seed(config.seed + progress["step"])
    model.train()
    start_time, recent_losses = time.time(), []
    for step in range(progress["step"] + 1, config.max_steps + 1):
        loss = loss_on(model, *get_batch(train_data, config.context_len, config.batch_size, batch_generator))
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        recent_losses.append(loss.item())

        if step % config.eval_interval == 0:
            val_loss = validation_loss(model, config)
            progress["history"].append({"step": step, "train_loss": sum(recent_losses) / len(recent_losses), "val_loss": val_loss})
            recent_losses = []
            if val_loss < progress["best_val_loss"]:
                progress.update(best_step=step, best_val_loss=val_loss)
                torch.save(model.state_dict(), best_path)
            progress["step"] = step
            progress["train_seconds"] += time.time() - start_time
            start_time = time.time()
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
                "batch_generator": batch_generator.get_state(),
                "progress": progress,
            }, last_path)
            print(f"step {step:5d} | train {progress['history'][-1]['train_loss']:.4f} | val {val_loss:.4f} "
                  f"| best {progress['best_val_loss']:.4f} @ {progress['best_step']}")

    model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=True))
    val_bpc = transformer_bpc(model, val_data, config.context_len)
    run_record = {
        "val_bpc": val_bpc,
        "config": asdict(config),
        "batch_size": config.batch_size,
        "num_params": model.num_params(),
        "device": torch.cuda.get_device_name() if DEVICE == "cuda" else "cpu",
        "torch_version": torch.__version__,
        **progress,
    }
    (RUNS_DIR / f"{config.run_name}.json").write_text(json.dumps(run_record, indent=2))
    return run_record

## 8. Reference condition 2: context-32 transformer, seed 0

In [ ]:
reference_run = train(Config(context_len=32, seed=0))
print(f"best validation loss {reference_run['best_val_loss']:.4f} at step {reference_run['best_step']}, "
      f"{reference_run['train_seconds'] / 60:.1f} min on {reference_run['device']}")